# AstroLongevity Data Pipeline (Google Colab / Kaggle)

**NASA Space Apps Challenge 2026**

This notebook fetches and processes transcriptomics data from the NASA Open Science Data Repository (OSDR). 
It employs production-grade data engineering practices: explicit HTTP error handling, disk-first streaming to prevent OOM errors, and strict type-checking validation gates.

Target Datasets:
- **OSD-21** (Microarray)
- **OSD-104** (RNA-Seq)
- **OSD-101** (RNA-Seq)

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import logging

# Initialize Production Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Create a local directory in Colab to store the downloaded files permanently
os.makedirs('nasa_data', exist_ok=True)
logger.info("Environment initialized. Created 'nasa_data' directory for secure local storage.")

## Step 1: Query NASA OSDR API for Study File Manifests
We query the official NASA API with strict exception handling.

In [ ]:
def get_study_files(osd_id):
    """Fetches the list of files available for a given study from NASA OSDR."""
    api_url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}"
    logger.info(f"Fetching file manifest for {osd_id}...")
    
    response = requests.get(api_url)
    
    # Explicit Error Handling
    if response.status_code != 200:
        raise requests.exceptions.HTTPError(f"Failed to fetch files for {osd_id}. HTTP Status: {response.status_code}")
    
    data = response.json()
    file_list = data.get('study_files', [])
    
    if not file_list:
        raise ValueError(f"No files found in the payload for {osd_id}.")
        
    return file_list

# Test it on OSD-104
osd104_files = get_study_files("OSD-104")
logger.info(f"SUCCESS: Found {len(osd104_files)} total files for OSD-104.")

## Step 2: Stream Download to Disk (Memory Management)
To prevent OOM crashes on massive transcriptomic files, we stream the file bytes directly to the local disk before ever loading them into pandas.

In [ ]:
def download_processed_data(osd_id, file_list):
    """Filters the file list and streams the processed data safely to disk."""
    target_file = None
    target_url = None
    
    # Search logic prioritizing exact string matches or known OSDR tag structures if possible
    for file_info in file_list:
        fname = file_info.get('file_name', '').lower()
        if (fname.endswith('.csv') or fname.endswith('.tsv') or fname.endswith('.txt')) and ('rna_seq' in fname or 'microarray' in fname) and ('normalized' in fname or 'differential' in fname):
            target_file = file_info.get('file_name')
            target_url = f"https://osdr.nasa.gov{file_info.get('remote_url')}"
            break
            
    if target_file is None:
        raise FileNotFoundError(f"Could not locate processed count matrix for {osd_id}.")
    
    logger.info(f"Target identified for {osd_id}: {target_file}")
    
    save_path = f"nasa_data/{osd_id}_{target_file}"
    logger.info(f"Streaming download from {target_url} to {save_path}...")
    
    # Stream to disk to prevent RAM exhaustion
    with requests.get(target_url, stream=True) as r:
        if r.status_code != 200:
            raise requests.exceptions.HTTPError(f"Failed to download data. HTTP Status: {r.status_code}")
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                
    logger.info(f"SUCCESS: Data securely saved to local disk at: {save_path}")
    
    # Now safely load from local disk into Pandas
    sep = '\t' if save_path.endswith('.tsv') or save_path.endswith('.txt') else ','
    df = pd.read_csv(save_path, sep=sep)
    
    return df, save_path

# NOTE: In Colab, you can uncomment the lines below to execute the download.
# df_104, saved_path_104 = download_processed_data("OSD-104", osd104_files)
# logger.info(f"Data for OSD-104 loaded into memory with shape {df_104.shape}")

## Step 3: Strict Validation & QC Gates
Before proceeding with signature reversal, we validate matrix dimensions, null counts, and explicit numeric casting.

In [ ]:
def validate_dataframe(df, dataset_name):
    """Runs mathematical validation gates and type checking on the DataFrame."""
    logger.info(f"--- Running QC Validation for {dataset_name} ---")
    
    # Gate 1: Check for empty dataframe
    row_count, col_count = df.shape
    if row_count <= 1000:
        raise ValueError(f"FAIL: Dataset {dataset_name} has only {row_count} rows. Expected >1000 genes.")
    if col_count <= 2:
        raise ValueError(f"FAIL: Dataset {dataset_name} has only {col_count} columns. Missing sample data.")
    logger.info(f"PASS: Matrix dimensions valid ({row_count} genes, {col_count} columns).")
    
    # Gate 2: Check for missing values in the index/genes
    first_col = df.columns[0]
    missing_genes = df[first_col].isna().sum()
    if missing_genes > 0:
        raise ValueError(f"FAIL: Dataset {dataset_name} contains {missing_genes} missing gene identifiers.")
    logger.info("PASS: No missing gene identifiers detected.")
    
    # Gate 3: Explicit numeric type checking for value columns
    # We assume columns 1 to N contain numeric count/logFC data
    numeric_cols = df.columns[1:]
    for col in numeric_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise TypeError(f"FAIL: Column '{col}' contains non-numeric data types (e.g., strings/objects).")
    logger.info("PASS: All sample/count columns are strictly numeric data types.")
    
    logger.info("STATUS: Data passed strict scientific validation. Ready for downstream pipeline.")

# In Colab, uncomment to run validation:
# validate_dataframe(df_104, "OSD-104")